In [ ]:
# DBA 콘솔 프로그램 interface

import csv
import shlex
import shutil
import sqlite3
from datetime import datetime
from pathlib import Path


DEFAULT_DB_NAMES = ["파리바게트.db", "paris_baguette.db", "database.db"]

def quote_identifier(identifier):
    escaped = identifier.replace('"', '""')
    return f'"{escaped}"'


# DB 파일 자동으로 찾기
def discover_db_paths():
    paths = []

    cwd = Path.cwd()
    home = Path.home()

    project_roots = [
        cwd,
        cwd.parent,
        home / "paris-baguette-system-db",
        home / "Desktop" / "paris-baguette-system-db",
    ]

    unique_roots = []
    for root in project_roots:
        if root not in unique_roots:
            unique_roots.append(root)

    for root in unique_roots:
        search_dirs = [
            root / "schema",
            root / "data",
            root,
        ]

        for base in search_dirs:
            for name in DEFAULT_DB_NAMES:
                candidate = base / name
                if candidate.exists() and candidate not in paths:
                    paths.append(candidate)

    return paths


# 사용할 DB 파일 선택
def select_db_path():
    found = discover_db_paths()

    if found:
        for path in found:
            if path.name == "파리바게트.db" and path.parent.name == "schema":
                print(f"DB 파일 자동 선택: {path}")
                return str(path)

        print(f"DB 파일 자동 선택: {found[0]}")
        return str(found[0])

    print("DB 파일을 자동으로 찾지 못했습니다.")
    print("예: schema/파리바게트.db")
    return input("사용할 DB 파일 경로를 입력하세요 > ").strip()


class DBAConsole:
    def __init__(self, db_path):
        self.db_path = Path(db_path)
        self.conn = None
        self.in_transaction = False
        self.last_result = None
        self.last_headers = None

    # 1. DB 연결
    def connect(self):
        if not self.db_path.exists():
            raise FileNotFoundError(f"DB 파일을 찾을 수 없습니다: {self.db_path}")

        self.conn = sqlite3.connect(self.db_path)
        self.conn.row_factory = sqlite3.Row
        self.conn.execute("PRAGMA foreign_keys = ON;")

        print(f"\nDB 연결 완료: {self.db_path}")

    # 2. DB 연결 종료
    def close(self):
        if self.conn:
            if self.in_transaction:
                print("저장하지 않은 작업이 있어 되돌리고 종료합니다.")
                self.conn.rollback()
                self.in_transaction = False

            self.conn.close()
            print("DB 연결을 종료했습니다.")

    # 3. 콘솔 메뉴 실행
    def run(self):
        self.connect()

        while True:
            self.print_menu()
            choice = input("메뉴 번호를 입력하세요 > ").strip()

            try:
                if choice == "1":
                    self.show_tables()
                elif choice == "2":
                    self.show_schema()
                elif choice == "3":
                    self.execute_sql_or_dot_command()
                elif choice == "4":
                    self.begin_transaction()
                elif choice == "5":
                    self.commit_transaction()
                elif choice == "6":
                    self.rollback_transaction()
                elif choice == "7":
                    self.show_indexes()
                elif choice == "8":
                    self.check_constraints()
                elif choice == "9":
                    self.show_row_counts()
                elif choice == "10":
                    self.export_select_to_csv()
                elif choice == "11":
                    self.backup_database()
                elif choice == "12":
                    self.add_index_or_unique_constraint()
                elif choice == "13":
                    self.export_dump()
                elif choice == "14":
                    self.restore_dump_to_new_db()
                elif choice == "0":
                    print("콘솔을 종료합니다.")
                    break
                else:
                    print("목록에 있는 번호를 입력해주세요.")
            except sqlite3.Error as e:
                print(f"SQLite 오류가 발생했습니다: {e}")
            except Exception as e:
                print(f"오류가 발생했습니다: {e}")

        self.close()

    # 4. 메뉴 출력
    @staticmethod
    def print_menu():
        print("\n" + "=" * 70)
        print("DBA 콘솔 메뉴")
        print("=" * 70)
        print("1. 테이블 목록 보기")
        print("2. 테이블 구조 보기")
        print("3. SQL 또는 도트 명령 실행")
        print("4. 작업 묶음 시작(BEGIN)")
        print("5. 작업 저장(COMMIT)")
        print("6. 작업 취소(ROLLBACK)")
        print("7. 인덱스 목록 보기")
        print("8. 제약조건/무결성 점검")
        print("9. 테이블별 행 개수 보기")
        print("10. SELECT 결과 CSV로 저장")
        print("11. DB 백업 만들기")
        print("12. 인덱스/UNIQUE 제약 추가")
        print("13. 데이터 덤프 SQL 저장")
        print("14. 덤프 SQL로 새 DB 복원")
        print("0. 종료")
        print("=" * 70)

    # 5. SELECT 결과 여러 행 가져오기
    def fetch_all(self, sql, params=()):
        cur = self.conn.execute(sql, params)
        return cur.fetchall()

    # 6. 조회 결과를 표처럼 출력
    @staticmethod
    def print_rows(rows):
        if not rows:
            print("조회 결과가 없습니다.")
            return

        headers = list(rows[0].keys())
        widths = []

        for header in headers:
            max_width = max(len(str(row[header])) for row in rows)
            widths.append(min(max(max_width, len(header), 8), 28))

        header_line = " | ".join(str(header).ljust(widths[i]) for i, header in enumerate(headers))
        print("\n" + header_line)
        print("-" * len(header_line))

        for row in rows:
            line = []
            for i, header in enumerate(headers):
                value = str(row[header])
                if len(value) > widths[i]:
                    value = value[: widths[i] - 3] + "..."
                line.append(value.ljust(widths[i]))
            print(" | ".join(line))

        print(f"\n총 {len(rows)}행")

    # 7. 최근 조회 결과 저장
    def remember_result(self, rows):
        if rows:
            self.last_result = rows
            self.last_headers = list(rows[0].keys())

    # 8. 테이블 목록 보기
    def show_tables(self):
        rows = self.fetch_all(
            """
            SELECT name AS table_name
            FROM sqlite_master
            WHERE type = 'table'
              AND name NOT LIKE 'sqlite_%'
            ORDER BY name;
            """
        )

        self.print_rows(rows)
        self.remember_result(rows)

    # 9. 테이블 구조 보기
    def show_schema(self):
        table_name = input("구조를 확인할 테이블 이름을 입력하세요. 전체는 Enter > ").strip()

        if not table_name:
            self.show_all_create_sql()
            return

        table = quote_identifier(table_name)

        print(f"\n[{table_name}] 컬럼 정보")
        rows = self.fetch_all(f"PRAGMA table_info({table});")
        self.print_rows(rows)

        print(f"\n[{table_name}] 외래키 정보")
        fk_rows = self.fetch_all(f"PRAGMA foreign_key_list({table});")
        self.print_rows(fk_rows)

        print(f"\n[{table_name}] CREATE SQL")
        create_rows = self.fetch_all(
            """
            SELECT sql AS create_sql
            FROM sqlite_master
            WHERE type = 'table'
              AND name = ?;
            """,
            (table_name,),
        )
        self.print_rows(create_rows)
        self.remember_result(create_rows)

    # 10. 전체 CREATE SQL 보기
    def show_all_create_sql(self):
        rows = self.fetch_all(
            """
            SELECT type, name, tbl_name, sql
            FROM sqlite_master
            WHERE type IN ('table', 'index', 'trigger', 'view')
              AND name NOT LIKE 'sqlite_%'
            ORDER BY type, name;
            """
        )

        self.print_rows(rows)
        self.remember_result(rows)

    # 11. SQL 또는 도트 명령 실행
    def execute_sql_or_dot_command(self):
        print("\n실행할 SQL 또는 도트 명령을 입력하세요.")
        print("예: SELECT * FROM 상품 LIMIT 5")
        print("예: .tables / .schema 상품 / .indexes / .dump / .read 파일명 / .help")
        print("SQL을 여러 줄로 입력할 때는 마지막 줄에 ; 만 입력하면 됩니다.")
        print("아무것도 입력하지 않고 Enter를 누르면 취소됩니다.")

        first_line = input("SQL 또는 명령 > ").strip()

        if not first_line:
            print("입력을 취소했습니다.")
            return

        if first_line.startswith("."):
            self.handle_dot_command(first_line)
            return

        lines = [first_line]

        while True:
            line = input("SQL > ")

            if line.strip() == ";":
                break

            lines.append(line)

        sql = "\n".join(lines).strip()

        if not sql:
            print("실행할 SQL이 없습니다.")
            return

        self.run_sql(sql)

    # 12. SQL 실행
    def run_sql(self, sql):
        cur = self.conn.cursor()
        cur.execute(sql)

        # 조회 쿼리면 결과를 출력하고, 수정 쿼리면 영향받은 행 수를 출력
        if sql.lstrip().lower().startswith(("select", "pragma", "with")):
            rows = cur.fetchall()
            self.print_rows(rows)
            self.remember_result(rows)
        else:
            if not self.in_transaction:
                self.conn.commit()
                print("SQL 실행 완료. 변경 내용이 바로 저장되었습니다.")
            else:
                print("SQL 실행 완료. COMMIT을 해야 최종 저장됩니다.")

            print(f"영향받은 행 수: {cur.rowcount}")

    # 13. 도트 명령 처리
    def handle_dot_command(self, command):
        try:
            parts = shlex.split(command)
        except ValueError:
            print("명령 형식을 다시 확인해주세요.")
            return

        if not parts:
            return

        cmd = parts[0].lower()
        args = parts[1:]

        if cmd == ".help":
            self.print_dot_help()
        elif cmd == ".tables":
            self.show_tables()
        elif cmd == ".schema":
            if args:
                self.print_schema_sql(args[0])
            else:
                self.show_all_create_sql()
        elif cmd == ".indexes":
            table_name = args[0] if args else None
            self.show_indexes(table_name)
        elif cmd == ".dump":
            filename = args[0] if args else None
            self.export_dump(filename)
        elif cmd == ".read":
            if not args:
                print(".read 뒤에 SQL 파일명을 입력해야 합니다.")
                return
            self.read_sql_file(args[0])
        else:
            print("지원하지 않는 도트 명령입니다. .help를 입력하면 목록을 볼 수 있습니다.")

    # 14. 도트 명령 도움말
    @staticmethod
    def print_dot_help():
        print("\n[사용 가능한 도트 명령]")
        print(".tables              : 테이블 목록 보기")
        print(".schema              : 전체 스키마 보기")
        print(".schema 테이블명      : 특정 테이블 스키마 보기")
        print(".indexes             : 전체 인덱스 보기")
        print(".indexes 테이블명     : 특정 테이블 인덱스 보기")
        print(".dump                : 현재 DB를 SQL 파일로 저장")
        print(".dump 파일명.sql      : 지정한 파일명으로 SQL 덤프 저장")
        print(".read 파일명.sql      : SQL 파일을 현재 DB에 실행")
        print(".help                : 도트 명령 도움말 보기")

    # 15. 특정 테이블 CREATE SQL 보기
    def print_schema_sql(self, table_name):
        rows = self.fetch_all(
            """
            SELECT type, name, tbl_name, sql
            FROM sqlite_master
            WHERE tbl_name = ?
              AND name NOT LIKE 'sqlite_%'
            ORDER BY type, name;
            """,
            (table_name,),
        )

        self.print_rows(rows)
        self.remember_result(rows)

    # 16. 작업 묶음 시작
    def begin_transaction(self):
        if self.in_transaction:
            print("이미 작업 묶음이 시작된 상태입니다.")
            return

        self.conn.execute("BEGIN IMMEDIATE;")
        self.in_transaction = True
        print("작업 묶음을 시작했습니다.")

    # 17. 작업 저장
    def commit_transaction(self):
        if not self.in_transaction:
            print("저장할 작업 묶음이 없습니다.")
            return

        self.conn.commit()
        self.in_transaction = False
        print("작업 내용을 저장했습니다.")

    # 18. 작업 취소
    def rollback_transaction(self):
        if not self.in_transaction:
            print("취소할 작업 묶음이 없습니다.")
            return

        self.conn.rollback()
        self.in_transaction = False
        print("작업 내용을 취소했습니다.")

    # 19. 인덱스 목록 보기
    def show_indexes(self, table_name=None):
        params = []
        where_table = ""

        if table_name:
            where_table = "AND tbl_name = ?"
            params.append(table_name)

        rows = self.fetch_all(
            f"""
            SELECT tbl_name AS table_name,
                   name AS index_name,
                   sql AS create_sql
            FROM sqlite_master
            WHERE type = 'index'
              AND name NOT LIKE 'sqlite_%'
              {where_table}
            ORDER BY tbl_name, name;
            """,
            params,
        )

        self.print_rows(rows)
        self.remember_result(rows)

    # 20. 제약조건/무결성 점검
    def check_constraints(self):
        print("\n[외래키 오류 확인]")
        self.conn.execute("PRAGMA foreign_keys = ON;")
        fk_rows = self.fetch_all("PRAGMA foreign_key_check;")

        if not fk_rows:
            print("외래키 오류가 없습니다.")
        else:
            print("외래키 오류가 발견되었습니다.")
            self.print_rows(fk_rows)

        print("\n[DB 무결성 확인]")
        integrity_rows = self.fetch_all("PRAGMA integrity_check;")
        self.print_rows(integrity_rows)

        print("\n[테이블별 외래키 개수]")
        tables = self.get_table_names()
        output = []

        for table_name in tables:
            fk_count = len(self.fetch_all(f"PRAGMA foreign_key_list({quote_identifier(table_name)});"))
            output.append({"table_name": table_name, "foreign_key_count": fk_count})

        print_rows_from_dict(output)

    # 21. 테이블명 목록 가져오기
    def get_table_names(self):
        rows = self.fetch_all(
            """
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
              AND name NOT LIKE 'sqlite_%'
            ORDER BY name;
            """
        )

        return [row["name"] for row in rows]

    # 22. 테이블별 행 개수 보기
    def show_row_counts(self):
        output = []

        for table_name in self.get_table_names():
            count = self.fetch_all(
                f"SELECT COUNT(*) AS row_count FROM {quote_identifier(table_name)};"
            )[0]["row_count"]

            output.append(
                {
                    "table_name": table_name,
                    "row_count": count,
                }
            )

        if not output:
            print("테이블이 없습니다.")
            return

        print_rows_from_dict(output)
        self.last_result = dicts_to_rows_like(output)
        self.last_headers = ["table_name", "row_count"]

    # 23. SELECT 결과 CSV 저장
    def export_select_to_csv(self):
        print("\nCSV로 저장할 SELECT SQL을 입력하세요.")
        print("끝낼 때는 ; 만 입력하면 됩니다.")

        lines = []

        while True:
            line = input("SELECT SQL > ")

            if line.strip() == ";":
                break

            lines.append(line)

        sql = "\n".join(lines).strip()

        if not sql.lower().startswith(("select", "with")):
            print("CSV 저장은 SELECT 또는 WITH 쿼리만 가능합니다.")
            return

        filename = input("저장할 CSV 파일명 입력, 비워두면 자동 생성 > ").strip()

        if not filename:
            filename = f"query_result_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

        cur = self.conn.execute(sql)
        rows = cur.fetchall()

        if not rows:
            print("조회 결과가 없어 CSV 파일을 만들지 않았습니다.")
            return

        self.write_rows_to_csv(rows, filename)

    # 24. 조회 결과를 CSV 파일로 저장
    def write_rows_to_csv(self, rows, filename):
        with open(filename, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.writer(f)
            writer.writerow(rows[0].keys())

            for row in rows:
                writer.writerow([row[key] for key in row.keys()])

        print(f"CSV 저장 완료: {filename}")

    # 25. DB 백업 만들기
    def backup_database(self):
        backup_dir = Path("backup")
        backup_dir.mkdir(exist_ok=True)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_path = backup_dir / f"{self.db_path.stem}_backup_{timestamp}{self.db_path.suffix}"

        shutil.copy2(self.db_path, backup_path)

        print(f"DB 백업 완료: {backup_path}")

    # 26. 인덱스 또는 UNIQUE 제약 추가
    def add_index_or_unique_constraint(self):
        print("\n인덱스 또는 UNIQUE 제약을 추가합니다.")
        print("일반 인덱스는 검색 속도 향상에 사용하고, UNIQUE 인덱스는 중복을 막는 용도로 사용할 수 있습니다.")
        print("CHECK, FOREIGN KEY 같은 제약은 SQLite 특성상 기존 테이블에 바로 추가하기 어렵습니다.")

        table_name = input("테이블 이름을 입력하세요 > ").strip()
        columns = input("컬럼 이름을 입력하세요. 여러 개면 쉼표로 구분 > ").strip()
        index_name = input("인덱스 이름 입력, 비워두면 자동 생성 > ").strip()
        unique = input("중복을 막는 UNIQUE 인덱스로 만들까요? (y/N) > ").strip().lower() == "y"

        if not table_name or not columns:
            print("테이블 이름과 컬럼 이름을 입력해야 합니다.")
            return

        column_list = [col.strip() for col in columns.split(",") if col.strip()]

        if not column_list:
            print("컬럼 이름을 입력해야 합니다.")
            return

        if not index_name:
            index_name = f"idx_{table_name}_{'_'.join(column_list)}"

        quoted_table = quote_identifier(table_name)
        quoted_index = quote_identifier(index_name)
        quoted_columns = ", ".join(quote_identifier(col) for col in column_list)
        unique_sql = "UNIQUE " if unique else ""

        sql = f"CREATE {unique_sql}INDEX IF NOT EXISTS {quoted_index} ON {quoted_table} ({quoted_columns});"

        print("\n실행할 SQL:")
        print(sql)

        confirm = input("이 SQL을 실행할까요? (y/N) > ").strip().lower()

        if confirm != "y":
            print("인덱스 추가를 취소했습니다.")
            return

        self.conn.execute(sql)

        if not self.in_transaction:
            self.conn.commit()

        if unique:
            print("UNIQUE 인덱스가 추가되었습니다.")
        else:
            print("인덱스가 추가되었습니다.")

    # 27. 데이터 덤프 SQL 저장
    def export_dump(self, filename=None):
        dump_dir = Path("dump")
        dump_dir.mkdir(exist_ok=True)

        if not filename:
            filename = f"{self.db_path.stem}_dump_{datetime.now().strftime('%Y%m%d_%H%M%S')}.sql"

        dump_path = Path(filename)

        if not dump_path.is_absolute():
            dump_path = dump_dir / dump_path

        with open(dump_path, "w", encoding="utf-8") as f:
            for line in self.conn.iterdump():
                f.write(f"{line}\n")

        print(f"데이터 덤프 저장 완료: {dump_path}")

    # 28. 덤프 SQL로 새 DB 복원
    def restore_dump_to_new_db(self):
        dump_file = input("복원할 SQL 덤프 파일 경로를 입력하세요 > ").strip()

        if not dump_file:
            print("복원할 파일을 입력하지 않아 취소했습니다.")
            return

        dump_path = Path(dump_file)

        if not dump_path.exists():
            print("해당 SQL 파일을 찾을 수 없습니다.")
            return

        restore_dir = Path("restore")
        restore_dir.mkdir(exist_ok=True)

        output_name = input("새로 만들 DB 파일명 입력, 비워두면 자동 생성 > ").strip()

        if not output_name:
            output_name = f"{self.db_path.stem}_restored_{datetime.now().strftime('%Y%m%d_%H%M%S')}.db"

        output_path = Path(output_name)

        if not output_path.is_absolute():
            output_path = restore_dir / output_path

        if output_path.exists():
            print("같은 이름의 DB 파일이 이미 있습니다. 다른 이름을 사용해주세요.")
            return

        sql_text = dump_path.read_text(encoding="utf-8")

        restore_conn = sqlite3.connect(output_path)

        try:
            restore_conn.executescript(sql_text)
            restore_conn.commit()
        finally:
            restore_conn.close()

        print(f"새 DB 복원 완료: {output_path}")
        print("현재 연결된 원본 DB는 변경되지 않았습니다.")

    # 29. SQL 파일을 현재 DB에 실행
    def read_sql_file(self, filename):
        sql_path = Path(filename)

        if not sql_path.exists():
            print("해당 SQL 파일을 찾을 수 없습니다.")
            return

        print("이 기능은 SQL 파일 내용을 현재 DB에 실행합니다.")
        print("현재 DB가 바뀔 수 있으니 백업 후 사용하는 것을 권장합니다.")
        confirm = input("계속 진행할까요? (y/N) > ").strip().lower()

        if confirm != "y":
            print("SQL 파일 실행을 취소했습니다.")
            return

        sql_text = sql_path.read_text(encoding="utf-8")

        self.conn.executescript(sql_text)

        if not self.in_transaction:
            self.conn.commit()

        print("SQL 파일 실행이 완료되었습니다.")


# 딕셔너리 리스트를 표처럼 출력
def print_rows_from_dict(rows):
    if not rows:
        print("조회 결과가 없습니다.")
        return

    headers = list(rows[0].keys())
    widths = []

    for header in headers:
        max_width = max(len(str(row[header])) for row in rows)
        widths.append(min(max(max_width, len(header), 8), 28))

    header_line = " | ".join(headers[i].ljust(widths[i]) for i in range(len(headers)))
    print("\n" + header_line)
    print("-" * len(header_line))

    for row in rows:
        line = []
        for i, header in enumerate(headers):
            value = str(row[header])
            if len(value) > widths[i]:
                value = value[: widths[i] - 3] + "..."
            line.append(value.ljust(widths[i]))
        print(" | ".join(line))

    print(f"\n총 {len(rows)}행")


# 딕셔너리 결과도 CSV 저장이 가능하게 맞추기
class DictRow(dict):
    def keys(self):
        return super().keys()


# 딕셔너리 리스트를 Row 비슷한 형태로 바꾸기
def dicts_to_rows_like(rows):
    return [DictRow(row) for row in rows]


# 프로그램 시작
def main():
    print("DBA 콘솔을 시작합니다.")

    db_path = select_db_path()

    if not db_path:
        print("DB 파일을 선택하지 않아 종료합니다.")
        return

    console = DBAConsole(db_path)
    console.run()


if __name__ == "__main__":
    main()


DBA 콘솔을 시작합니다.
DB 파일 자동 선택: /Users/kimmiso/paris-baguette-system-db/schema/파리바게트.db

DB 연결 완료: /Users/kimmiso/paris-baguette-system-db/schema/파리바게트.db

DBA 콘솔 메뉴
1. 테이블 목록 보기
2. 테이블 구조 보기
3. SQL 또는 도트 명령 실행
4. 작업 묶음 시작(BEGIN)
5. 작업 저장(COMMIT)
6. 작업 취소(ROLLBACK)
7. 인덱스 목록 보기
8. 제약조건/무결성 점검
9. 테이블별 행 개수 보기
10. SELECT 결과 CSV로 저장
11. DB 백업 만들기
12. 인덱스/UNIQUE 제약 추가
13. 데이터 덤프 SQL 저장
14. 덤프 SQL로 새 DB 복원
0. 종료


메뉴 번호를 입력하세요 >  1



table_name
----------
견과류       
고객        
공급        
공급내역      
공급업체      
동의내역      
발주        
발주상세      
보유        
브랜드       
비회원       
빵         
상품        
샐러드       
스낵        
아동음료류     
양갱류       
원두커피류     
유제품류      
음료        
젤리류       
지점        
직원        
초코과자류     
취급        
케이크       
쿠키류       
탄산음료류     
판매        
판매상세      
회원        

총 31행

DBA 콘솔 메뉴
1. 테이블 목록 보기
2. 테이블 구조 보기
3. SQL 또는 도트 명령 실행
4. 작업 묶음 시작(BEGIN)
5. 작업 저장(COMMIT)
6. 작업 취소(ROLLBACK)
7. 인덱스 목록 보기
8. 제약조건/무결성 점검
9. 테이블별 행 개수 보기
10. SELECT 결과 CSV로 저장
11. DB 백업 만들기
12. 인덱스/UNIQUE 제약 추가
13. 데이터 덤프 SQL 저장
14. 덤프 SQL로 새 DB 복원
0. 종료


메뉴 번호를 입력하세요 >  2
구조를 확인할 테이블 이름을 입력하세요. 전체는 Enter >  회원



[회원] 컬럼 정보

cid      | name     | type         | notnull  | dflt_value | pk      
---------------------------------------------------------------------
0        | 고객ID     | VARCHAR(20)  | 0        | None       | 1       
1        | 아이디      | VARCHAR(30)  | 1        | None       | 0       
2        | 비밀번호     | VARCHAR(100) | 0        | None       | 0       
3        | 연락처      | VARCHAR(20)  | 0        | None       | 0       
4        | 이메일      | VARCHAR(100) | 0        | None       | 0       
5        | 주소       | VARCHAR(100) | 0        | None       | 0       
6        | 가입일자     | TEXT         | 0        | None       | 0       
7        | 정보제공동의여부 | INTEGER      | 0        | None       | 0       

총 8행

[회원] 외래키 정보

id       | seq      | table    | from     | to       | on_update | on_delete | match   
---------------------------------------------------------------------------------------
0        | 0        | 고객       | 고객ID     | 고객ID     | NO ACTION | CASCADE   | NONE    

총 

...

새 DB 복원 완료: restore/파리바게트_restored_20260619_015503.db
현재 연결된 원본 DB는 변경되지 않았습니다.

DBA 콘솔 메뉴
1. 테이블 목록 보기
2. 테이블 구조 보기
3. SQL 또는 도트 명령 실행
4. 작업 묶음 시작(BEGIN)
5. 작업 저장(COMMIT)
6. 작업 취소(ROLLBACK)
7. 인덱스 목록 보기
8. 제약조건/무결성 점검
9. 테이블별 행 개수 보기
10. SELECT 결과 CSV로 저장
11. DB 백업 만들기
12. 인덱스/UNIQUE 제약 추가
13. 데이터 덤프 SQL 저장
14. 덤프 SQL로 새 DB 복원
0. 종료


메뉴 번호를 입력하세요 >  3



실행할 SQL 또는 도트 명령을 입력하세요.
예: SELECT * FROM 상품 LIMIT 5
예: .tables / .schema 상품 / .indexes / .dump / .read 파일명 / .help
SQL을 여러 줄로 입력할 때는 마지막 줄에 ; 만 입력하면 됩니다.
아무것도 입력하지 않고 Enter를 누르면 취소됩니다.


SQL 또는 명령 >  .help



[사용 가능한 도트 명령]
.tables              : 테이블 목록 보기
.schema              : 전체 스키마 보기
.schema 테이블명      : 특정 테이블 스키마 보기
.indexes             : 전체 인덱스 보기
.indexes 테이블명     : 특정 테이블 인덱스 보기
.dump                : 현재 DB를 SQL 파일로 저장
.dump 파일명.sql      : 지정한 파일명으로 SQL 덤프 저장
.read 파일명.sql      : SQL 파일을 현재 DB에 실행
.help                : 도트 명령 도움말 보기

DBA 콘솔 메뉴
1. 테이블 목록 보기
2. 테이블 구조 보기
3. SQL 또는 도트 명령 실행
4. 작업 묶음 시작(BEGIN)
5. 작업 저장(COMMIT)
6. 작업 취소(ROLLBACK)
7. 인덱스 목록 보기
8. 제약조건/무결성 점검
9. 테이블별 행 개수 보기
10. SELECT 결과 CSV로 저장
11. DB 백업 만들기
12. 인덱스/UNIQUE 제약 추가
13. 데이터 덤프 SQL 저장
14. 덤프 SQL로 새 DB 복원
0. 종료
